# Periodic Hubbard operators

The nearest-neighbor hopping term uses two periodic plaquette colors.
The on-site term phases matched up- and down-spin occupation modes.
Here are the five source signatures and declared immediate calls, followed
by an executable two-by-two example built with FlagQuantum.

Adapted from the [Qualtran v0.7.0 notebook](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/chemistry/trotter/hubbard/hubbard.ipynb), Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import Markdown, SVG, display

from flagquantum.algorithms.primitives.hubbard_resource_model import (
    source_hubbard_hopping_plaquette_cost,
    source_hubbard_hopping_plaquette_signature,
    source_hubbard_hopping_tile_cost,
    source_hubbard_hopping_tile_signature,
    source_hubbard_interaction_cost,
    source_hubbard_interaction_signature,
)
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.drawer.call_graph import (
    export_modeled_call_graph_dot,
    format_call_graph_markdown,
    render_call_graph_svg,
)
from flagquantum.drawer.signature_graph import export_signature_graph_dot


def show_case(signature, model):
    graph = modeled_call_graph(model, max_depth=1)
    for dot in (
        export_signature_graph_dot(signature, model.name, pretty=True),
        export_modeled_call_graph_dot(model, max_depth=1),
    ):
        display(SVG(data=render_call_graph_svg(dot).decode()))
    display(Markdown(format_call_graph_markdown(
        graph.root, graph.direct_calls, graph.leaf_counts, title=lambda node: node.name
    )))
    return {node.name: count for node, count in graph.leaf_counts.items()}

## Hopping tile

The eight-by-eight pink example at angle 0.5 declares 32 plaquette calls, including both spin sectors.

In [ ]:
signature = source_hubbard_hopping_tile_signature(8)
model = source_hubbard_hopping_tile_cost(8, 0.5)
assert show_case(signature, model) == {'HoppingPlaquette': 32}

## Hopping plaquette

The four-mode cycle declares four Fourier calls and two rotations. The source example passes 8 as the plaquette angle and 0.15 as its synthesis tolerance. The concrete cost model uses that angle and reports declared gate categories.

In [ ]:
signature = source_hubbard_hopping_plaquette_signature()
model = source_hubbard_hopping_plaquette_cost(8)
assert show_case(signature, model) == {'TwoBitFFFT': 4, 'Rz': 2}

## On-site interaction

The example has 64 on-site rotations, with source angle 0.5 times coupling 4. The through signature contains two opaque side-width words.

In [ ]:
signature = source_hubbard_interaction_signature(8)
model = source_hubbard_interaction_cost(8, 0.5, 4.0)
assert show_case(signature, model) == {'Rz': 64}

## Hopping tile with Hamming-weight phasing

The eight-by-eight tile at angle 0.15 declares 128 Fourier calls and two batches of 32 modes.

In [ ]:
signature = source_hubbard_hopping_tile_signature(8)
model = source_hubbard_hopping_tile_cost(8, 0.15, hamming_weight=True)
assert show_case(signature, model) == {'TwoBitFFFT': 128, 'HammingWeightPhasing': 2}

## On-site interaction with Hamming-weight phasing

At angle 0.52728 and coupling 4, the declaration replaces the 64 rotations with two equal-exponent batches of 32 modes.

In [ ]:
signature = source_hubbard_interaction_signature(8)
model = source_hubbard_interaction_cost(8, 0.52728, 4.0, hamming_weight=True)
assert show_case(signature, model) == {'HammingWeightPhasing': 2}

## Run a hopping color and an interaction

The executable binders take physical row-major spin arrays. At side length
two each array contains four occupation wires. The source declarations shown
above contain two words of side width and describe the resource model.
The two native Hamming-weight operations reuse the same three clean work wires.
The reported model peak and emitted wire/gate counts describe different quantities.

In [ ]:
from flagquantum.algorithms.primitives.hubbard_lattice import (
    HubbardHoppingTile,
    HubbardInteraction,
)
from flagquantum.dynamic import DynamicCircuit
from flagquantum.qec.resources.estimation import estimate_ftqc_resources

binding = dict(
    length=2, up=(0, 1, 2, 3), down=(4, 5, 6, 7), timestep=0.23,
    hamming_weight=True, workspace=(8, 9, 10),
)
hopping = HubbardHoppingTile.make_on(**binding, hubb_t=0.8, pink=False)
interaction = HubbardInteraction.make_on(**binding, hubb_u=1.7)
circuit = DynamicCircuit(11).append(hopping).append(interaction)
ir = circuit.to_ir()
history_slots = [
    gate.metadata["classical_bit"] for gate in ir.instructions if gate.name == "measure"
]
assert history_slots == [0, 0, 1, 1]
assert sum(gate.name in {"t", "tdg"} for gate in ir.instructions) == 32
print("Emitted widths and counts:", ir.n_wires, 32, len(history_slots))
print("Interaction source-model peak:", estimate_ftqc_resources(interaction).logical.peak_logical_qubits)